# J1 — 강한 검증기 2단 구조 (FaithBench 사람 정답)

## 왜 하나

X2(11/13)에서 배포 판정기(F1 base + I2 조합)는 FaithBench 사람 정답 기준 AUROC 0.629로 판별력은 있었지만, **서비스 임계값 B에서 환각 응답의 11.8%만 잡았다**.
K1·K2에서 경량 판정기를 다시 학습해도 최신 LLM 탐지율은 +0.021에 그쳤다. 경량 판정기 하나로는 환각을 잡는 서비스가 되지 않는다.
그래서 **경량 판정기는 스트리밍 중 1단으로 두고, 응답이 끝나면 강한 검증기가 다시 판정하는 2단 구조**가 환각을 절반 이상 잡을 수 있는지 잰다.

## 무엇을 하나 (학습 없음)

- **후보(2단)**: `claude-haiku-5-5`, `claude-sonnet-5-5`(R4·M1과 같은 문장 단위 지시문, 문장마다 S/H/A, **H 문장이 하나라도 있으면 표시**),
  `Bespoke-MiniCheck-7B`(공개, 문장마다 근거 지지 확률, 응답 점수 = 최대 (1 − 확률), 임계값 = J-cal 멀쩡한 응답 등각 10%)
- **데이터**: FaithBench 주 분석 661개(X2와 같은 정답·제외 규칙). 근거를 `random.Random(2035)`로 섞어 절반은 **J-cal**(후보 선택·보정), 절반은 **J-test**(판정)
- **1단 거르기(보고·J1-3)**: 배포 조합 점수가 J-cal 환각 응답의 95%를 통과시키는 기준 이상인 응답만 2단으로 보낸다
- **devM**(K2a의 최신 Claude 응답, Opus 정답): 정답과 같은 지시문이라 Claude 후보에게 순환이 생겨 **보고만**

## 사전 정의 기준 (SSOT §6 J1, 실행 전 고정)

- **J1-0(유효성)**: Claude 후보마다 J-cal·J-test 응답의 JSON 파싱 성공 ≥ 95%. 미달 후보는 뺀다. MiniCheck가 설치·실행되지 않으면 후보에서 빠진다
- **선택(J-cal)**: 멀쩡한 응답 표시율 ≤ 15%인 후보 중 환각 탐지율 최대(동점이면 haiku → minicheck → sonnet). 없으면 J1 미달
- **J1-1(주, J-test)**: 선택 후보의 환각 응답 탐지율 ≥ 0.50, **그리고** 서비스 임계값 B 배포 조합 대비 탐지율 차이의 근거 단위 군집 부트스트랩 95% 하한 > 0 (2,000회, 시드 2035)
- **J1-2(J-test)**: 선택 후보의 멀쩡한 응답 표시율 ≤ 15%
- **J1 채택 = J1-1 그리고 J1-2** → 게이트웨이에 2단 검증을 넣는다(SPEC부터). 미달이면 "FaithBench급 환각은 현재 검증기로 절반을 못 잡는다"로 기록하고 서비스 주장을 줄인다
- **J1-3(구조)**: 1단 거르기 + 2단의 J-test 탐지율 ≥ 2단 단독 − 0.03이면 거르기를 쓰고, 아니면 모든 응답을 2단으로 보낸다
- **J1-4(지연)**: 선택 후보의 응답 1개 검증 p95 ≤ 5초(J-cal 40개, 실시간 요청)면 "응답을 잡아 두고 검증(hold)" 가능, 넘으면 응답 뒤 경고(비동기)로만 쓴다
- 보고만: 모든 후보의 J-cal·J-test 수치, H 또는 A 기준, 생성 모델별, devM, 비용. **API 비용 상한 $25**(제출 전 추정이 넘으면 멈춤)

> 런타임: **GPU**(MiniCheck-7B용, L4·A100 권장). 먼저 X2를 돌려 드라이브 `x2/x2_rows.csv`가 있어야 한다. Colab 비밀 값 `ANTHROPIC_API_KEY` 필요.
> 끊기면 다시 모두 실행(배치·점수는 드라이브 `j1/`에서 이어서). 끝나면 `j1_results.zip`(원문 없음)이 내려받아진다.


## 1. 환경 준비

In [ ]:
import os, sys, json, time, random, subprocess, shutil
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'anthropic', 'tiktoken', 'pysbd', 'scikit-learn'], check=True)
subprocess.run('git clone --depth 1 https://github.com/vectara/FaithBench.git 2>/dev/null || true', shell=True)
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
X2_DIR, K2_DIR, J1_DIR = os.path.join(OUT, 'x2'), os.path.join(OUT, 'k2'), os.path.join(OUT, 'j1')
os.makedirs(J1_DIR, exist_ok=True)
assert os.path.exists(os.path.join(X2_DIR, 'x2_rows.csv')), 'x2/x2_rows.csv 없음 — X2 노트북을 먼저 실행'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None
print('device', GPU)

## 2. FaithBench, X2 점수, J-cal·J-test 나누기

In [ ]:
# 판정 입력: RAGTruth train 응답의 문장 끝 + 8토큰 지점 (게이트웨이와 같은 cl100k 토큰, 문장 시작부터)
import pysbd, tiktoken
ENC = tiktoken.get_encoding('cl100k_base')
SEG = pysbd.Segmenter(language='en', clean=False, char_span=True)

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

def points_of(text, s=8):
    # (문장 시작 글자, 지점 글자 끝) 목록: 문장마다 s토큰째 + 문장 끝
    ids = ENC.encode(text, disallowed_special=())
    _, starts = ENC.decode_with_offsets(ids)
    ends = starts[1:] + [len(text)]
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        tok = [e for st, e in zip(starts, ends) if a <= st < sp.end or (st < a < e)]
        for k, e in enumerate(tok, 1):
            if k % s == 0 or k == len(tok):
                out.append((a, min(e, sp.end)))
    return out

def make_inputs(responses, sources, n_resp=300, n_pts=300, seed=2027):
    rs = random.Random(seed)
    pool = [r for r in responses if r['split'] == 'train']
    cands = []
    for r in rs.sample(pool, min(n_resp, len(pool))):
        ref = build_reference(sources[r['source_id']])
        for a, cut in points_of(r['response']):
            text = r['response'][a:cut].strip()
            if text: cands.append({'ref': ref, 'text': text, 'prior': r['response'][:a], 'task': sources[r['source_id']]['task_type']})
    return rs.sample(cands, min(n_pts, len(cands)))

import glob, re
FB = []
for f in sorted(glob.glob('FaithBench/data_for_release/batch_*.json'), key=lambda p: int(re.findall(r'(\d+)', p)[-1])):
    FB += json.load(open(f))['samples']
SEV = {'Benign': 1, 'Questionable': 2}
def sev(l): return 3 if l.startswith('Unwanted') else SEV.get(l, 0)

def fb_label(s):
    # 최악 등급(FaithBench binarize.py의 worst): 3 = Unwanted가 하나라도, 2 = Questionable, 1 = Benign만, 0 = 표시 없음
    ls = {l for a in s['annotations'] for l in a['label']}
    return max([sev(l) for l in ls], default=0)

def fb_onset(s):
    # 첫 Unwanted 구간의 요약 글자 위치 (없으면 None)
    st = [a['summary_start'] for a in s['annotations'] if any(l.startswith('Unwanted') for l in a['label']) and a.get('summary_start') is not None]
    return min(st) if st else None

def ngrams(t, n=8):
    w = re.findall(r'\w+', t.lower())
    return {tuple(w[i:i + n]) for i in range(max(len(w) - n + 1, 0))}

def overlap_flags(fb_sources, rt_sources, thr=0.5):
    # FaithBench 근거가 RAGTruth train 요약 근거와 겹치는지: 8-gram 포함률 ≥ 0.5
    rt = [ngrams(t) for t in rt_sources]
    out = {}
    for s in fb_sources:
        g = ngrams(s)
        out[s] = max([len(g & r) / max(len(g), 1) for r in rt], default=0.0) >= thr if g else False
    return out

import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

def auroc(y, s):
    y, s = np.asarray(y, bool), np.asarray(s, float)
    if y.all() or (~y).all(): return float('nan')
    return float(roc_auc_score(y, s))

def x2_rows(FB, SC, thr, overlap):
    # 요약 하나 = 행. hal = Unwanted 있음, clean = 표시 없음 또는 Benign만, Questionable이 최악이면 주 분석에서 뺌
    rows = []
    sid = {t: k for k, t in enumerate(sorted({x['source'] for x in FB}))}
    for i, s in enumerate(FB):
        pts = SC.get(str(i), [])
        lab = fb_label(s); on = fb_onset(s)
        r = {'i': i, 'src': sid[s['source']], 'summarizer': s['metadata']['summarizer'], 'worst': lab,
             'hal': lab == 3, 'primary': lab != 2 and not overlap.get(s['source'], False), 'overlap': overlap.get(s['source'], False),
             'n_pts': len(pts), 'onset': on,
             'hhem21': 1 - s['metadata']['hhem-2.1'], 'hhem21_en': 1 - s['metadata']['hhem-2.1-english']}
        for b in ['gpt-3.5-turbo', 'gpt-4-turbo', 'gpt_4o', 'trueteacher', 'true_nli']:
            v = s['metadata'].get(b); r[f'pred_{b}'] = None if v is None else int(v == 0)      # 1 = 환각이라고 예측
        for name, col in [('c', 4), ('f', 2), ('i', 3), ('n', 5)]:
            vals = [p[col] for p in pts if len(p) > col and p[col] is not None]
            r[f'max_{name}'] = max(vals) if vals else np.nan
        for name, col in [('c', 4), ('n', 5)]:
            t = thr.get(name)
            if t is None or np.isnan(r[f'max_{name}']): continue
            first = next((p[0] for p in pts if p[col] is not None and p[col] >= t), None)
            r[f'flag_{name}'] = first is not None
            h0 = max([p[0] for p in pts if p[0] <= on], default=0) if on is not None else None     # 첫 Unwanted 구간이 든 문장의 시작
            r[f'strict_{name}'] = bool(on is not None and first is not None and first >= h0)
        rows.append(r)
    return pd.DataFrame(rows)

def balanced_acc(y, pred):
    y, p = np.asarray(y, bool), np.asarray(pred, bool)
    return float((p[y].mean() + (~p[~y]).mean()) / 2)

def cluster_boot(df, fn, n=2000, seed=2034):
    # 같은 근거의 요약 10개가 묶여 있으므로 근거 단위로 다시 뽑는다
    srcs = df.src.unique(); rng = np.random.default_rng(seed)
    g = {s: df[df.src == s] for s in srcs}
    vals = []
    for _ in range(n):
        d = pd.concat([g[s] for s in rng.choice(srcs, len(srcs))])
        v = fn(d)
        if v == v: vals.append(v)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def x2_summary(df):
    P = df[df.primary]
    out = {'n_all': len(df), 'n_primary': len(P), 'n_hal': int(P.hal.sum()), 'n_clean': int((~P.hal).sum()),
           'n_overlap': int(df.overlap.sum()), 'n_questionable_worst': int((df.worst == 2).sum())}
    cols = [c for c in ['max_c', 'max_f', 'max_i', 'max_n', 'hhem21', 'hhem21_en'] if c in P and P[c].notna().all()]
    out['auroc'] = {c: auroc(P.hal, P[c]) for c in cols}
    out['auroc_ci'] = {c: cluster_boot(P, lambda d, c=c: auroc(d.hal, d[c])) for c in cols}
    out['auroc_diff_c_vs_hhem21'] = {'delta': out['auroc']['max_c'] - out['auroc']['hhem21'],
                                     'ci': cluster_boot(P, lambda d: auroc(d.hal, d.max_c) - auroc(d.hal, d.hhem21))}
    if 'max_n' in cols:
        out['auroc_diff_n_vs_c'] = {'delta': out['auroc']['max_n'] - out['auroc']['max_c'],
                                    'ci': cluster_boot(P, lambda d: auroc(d.hal, d.max_n) - auroc(d.hal, d.max_c))}
    out['bacc'] = {}
    for c in ['flag_c', 'flag_n'] + [k for k in P.columns if k.startswith('pred_')]:
        if c in P and P[c].notna().all(): out['bacc'][c] = balanced_acc(P.hal, P[c].astype(bool))
    for c in ['flag_c', 'flag_n']:
        if c in P and P[c].notna().all():
            out[f'{c}_hal_rate'] = float(P[P.hal][c].astype(bool).mean()); out[f'{c}_clean_rate'] = float(P[~P.hal][c].astype(bool).mean())
    for c in ['strict_c', 'strict_n']:
        if c in P and P[c].notna().all(): out[f'{c}_rate'] = float(P[P.hal][c].astype(bool).mean())
    out['by_summarizer_auroc_c'] = {m: auroc(d.hal, d.max_c) for m, d in P.groupby('summarizer')}
    out['by_summarizer_n_hal'] = {m: int(d.hal.sum()) for m, d in P.groupby('summarizer')}
    Q = df[~df.overlap]                                  # 보고: Questionable도 환각으로
    out['auroc_q_as_hal'] = {c: auroc(Q.worst >= 2, Q[c]) for c in cols}
    return out

import pandas as pd
CANDS_CLAUDE = {'haiku': 'claude-haiku-5-5', 'sonnet': 'claude-sonnet-5-5'}
ORDER = ['haiku', 'minicheck', 'sonnet']          # 동점이면 싼 쪽 (사전 고정)
BUDGET_USD = 25.0                                  # 이 노트북 전체 API 상한 (사전 고정)

def j1_split(X2R, seed=2035):
    # 주 분석 행(X2와 같은 정의)만. 근거 단위로 섞어 앞 절반 = J-cal, 나머지 = J-test
    P = X2R[X2R.primary].copy()
    srcs = sorted(P.src.unique()); random.Random(seed).shuffle(srcs)
    cal = set(srcs[:len(srcs) // 2])
    P['part'] = ['cal' if s in cal else 'test' for s in P.src]
    assert not set(P[P.part == 'cal'].src) & set(P[P.part == 'test'].src), 'J-cal·J-test 근거가 겹침'
    return P

assert len(FB) == 750, len(FB)
X2R = pd.read_csv(os.path.join(X2_DIR, 'x2_rows.csv'))
assert len(X2R) == 750 and 'flag_cB' in X2R, 'X2 결과에 임계값 B 표시(flag_cB)가 없음'
P = j1_split(X2R)
for part, d in P.groupby('part'):
    print(f'J-{part}: 응답 {len(d)} (환각 {int(d.hal.sum())}, 멀쩡 {int((~d.hal).sum())}), 근거 {d.src.nunique()}')

## 3. Claude 후보 — 비용 추정, 배치 제출·결과 받기

In [ ]:
SYSTEM = '''You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.
You receive a REFERENCE that a language model was given, and the model's RESPONSE split into numbered sentences.
Label EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).
- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;
meta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.

Labels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),
"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

TASK_DESC = {'QA': 'QA — answer the question using the passages',
             'Summary': 'Summary — summarize the news article',
             'Data2txt': 'Data2txt — write an overview of a business from structured data and user reviews'}

def sentences(text):
    # (시작, 끝, 문장) — 판정 지점(points_of)과 같은 pysbd 분할
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        out.append((a, sp.start + len(sp.sent.rstrip()), sp.sent.strip()))
    return out

def label_msg(c, text):
    ss = sentences(text)
    lines = '\n'.join(f'[{i + 1}] {s}' for i, (_, _, s) in enumerate(ss))
    return f"TASK: {TASK_DESC[c['task']]}\n\nREFERENCE:\n<<<\n{c['ref_label']}\n>>>\n\nRESPONSE ({len(ss)} numbered sentences):\n{lines}"

def label_params(c, text):
    # R4와 같은 요청 (모델·max_tokens·지시문)
    return {'model': LABELER, 'max_tokens': 4096, 'system': SYSTEM, 'messages': [{'role': 'user', 'content': label_msg(c, text)}]}

def parse(text, n):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'type': e.get('type', ''), 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '')}
        return labs, len(labs) == n
    except Exception:
        return {}, False

def score_response(jF, jI, refs, text, fn):
    # 판정 지점(문장마다 8토큰째 + 문장 끝)마다 (문장 시작, 지점 끝, F1 base, I2, 조합)
    out = []
    for a, cut in points_of(text):
        t = text[a:cut].strip()
        if not t: continue
        sf, si = fn(jF, jI, refs[0], refs[1], {'text': t, 'prior': text[:a]})
        out.append((a, cut, sf, si, combine(sf, si)))
    return out

def gen_status(g):
    if g is None: return 'nogen'
    if g['stop'] == 'refusal': return 'refusal'
    if not g['text'].strip() or not sentences(g['text']): return 'empty'
    return 'valid'

import pandas as pd

def wilson(k, n, z=1.96):
    if n == 0: return float('nan'), float('nan'), float('nan')
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

def conformal(maxes, alpha=0.10):
    # cal-L 노트북(conformal_flag)과 같은 등각 보정: 멀쩡한 응답 최대 점수의 ceil((n+1)(1-α))번째 + 1e-6
    mx = np.sort(np.asarray(maxes, float)); n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

def response_rows(cases, gens, labels, scores, thr_c, thr_f):
    rows = []
    for cid, c in cases.items():
        g = gens.get(cid); st = gen_status(g)
        r = {'cid': cid, 'sid': c['sid'], 'task': c['task'], 'gen': c['gen'], 'flow': c['flow'], 'gold_in': c['gold_in'],
             'stop': g['stop'] if g else None, 'status': st}
        if st == 'valid':
            ss = sentences(g['text']); lab = labels.get(cid)
            r['status'] = 'ok' if lab is not None else 'nolabel'
            pts = scores.get(cid, [])
            r['n_sent'], r['n_pts'] = len(ss), len(pts)
            r['max_c'] = max([p[4] for p in pts], default=-np.inf)
            r['max_f'] = max([p[2] for p in pts], default=-np.inf)
            r['flag_c'], r['flag_f'] = bool(r['max_c'] >= thr_c), bool(r['max_f'] >= thr_f)
            first = next((p[0] for p in pts if p[4] >= thr_c), None)
            r['first_alarm_a'] = first
            if lab is not None:
                ls = [lab[i + 1]['label'] for i in range(len(ss))]
                r['n_H'], r['n_A'] = ls.count('H'), ls.count('A')
                r['clean'], r['strict_clean'] = r['n_H'] == 0, all(l == 'S' for l in ls)
                hs = [ss[i][0] for i, l in enumerate(ls) if l == 'H']
                r['first_H_a'] = hs[0] if hs else None
                r['strict_det'] = bool(hs and first is not None and first >= hs[0])
        rows.append(r)
    return pd.DataFrame(rows)

def rate_line(df, col='flag_c'):
    k, n = int(df[col].sum()), len(df)
    p, lo, hi = wilson(k, n)
    return {'k': k, 'n': n, 'rate': p, 'lo': lo, 'hi': hi}

def fmt(d):
    return f"{d['rate']:.3f} [{d['lo']:.3f}, {d['hi']:.3f}] ({d['k']}/{d['n']})" if d['n'] else '—'

def report(df, thr_name=''):
    ok = df[df.status == 'ok'].copy()
    for col in ['clean', 'strict_clean', 'flag_c', 'flag_f', 'strict_det']:
        ok[col] = ok[col].astype(bool)
    clean, hal = ok[ok.clean], ok[~ok.clean]
    out = {'status': df.status.value_counts().to_dict(), 'max_tokens_stops': int((df.stop == 'max_tokens').sum()),
           'clean_combo': rate_line(clean), 'clean_f1': rate_line(clean, 'flag_f'),
           'strict_clean_combo': rate_line(ok[ok.strict_clean]),
           'hal_any': rate_line(hal), 'hal_strict': rate_line(hal, 'strict_det'),
           'flag_precision_H': rate_line(ok[ok.flag_c], 'clean').copy(),
           'by_gen': {g: rate_line(d) for g, d in clean.groupby('gen')},
           'by_task': {t: rate_line(d) for t, d in clean.groupby('task')},
           'by_gen_task': {f'{g}|{t}': rate_line(d) for (g, t), d in clean.groupby(['gen', 'task'])},
           'hal_rate_by_gen': {g: float((~d.clean).mean()) for g, d in ok.groupby('gen')}}
    fp = out['flag_precision_H']                     # 표시된 응답 중 멀쩡한 비율 → H 비율로 바꿈
    if fp['n']: fp.update(k=fp['n'] - fp['k'], rate=1 - fp['rate'], lo=1 - fp['hi'], hi=1 - fp['lo'])
    qa = df[(df.flow == 'collection') & (df.gen == 'haiku')]
    out['qa_gold_in'] = float(qa.gold_in.astype(float).mean()) if len(qa) else None
    return out

LABELER = 'claude-opus-5-5'          # label_params 기본값. v_params가 후보 모델로 바꾼다

def fb_case(s):
    return {'task': 'Summary', 'ref_label': s['source']}

def v_params(c, text, model):
    # R4·M1과 같은 문장 단위 지시문, 모델만 바꾼다
    p = label_params(c, text); p['model'] = model
    return p

def est_cost(reqs, out_tokens=400):
    # 요청 글자 수 / 3.5 ≈ 입력 토큰 (추정). 배치 단가
    tot = 0.0
    for p in reqs.values():
        n_in = (len(p.get('system', '')) + sum(len(m['content']) for m in p['messages'])) / 3.5
        pin, pout = PRICE[p['model']]
        tot += n_in / 1e6 * pin + out_tokens / 1e6 * pout
    return tot

def fb_requests(P, FB):
    return {m: {f'{m}-{i}': v_params(fb_case(FB[i]), FB[i]['summary'], model) for i in P.i} for m, model in CANDS_CLAUDE.items()}

def claude_cols(P, FB, RES):
    # 응답마다: 파싱 성공, H 문장 하나라도(주), H 또는 A(보고). 파싱 실패 = 표시 안 함(탐지에 불리한 쪽)
    for m in CANDS_CLAUDE:
        ok_l, h_l, ha_l = [], [], []
        for i in P.i:
            r = RES[m].get(f'{m}-{i}'); n = len(sentences(FB[i]['summary']))
            labs, ok = parse(r['text'], n) if r else ({}, False)
            ls = [v['label'] for v in labs.values()] if ok else []
            ok_l.append(ok); h_l.append('H' in ls); ha_l.append('H' in ls or 'A' in ls)
        P[f'ok_{m}'], P[f'v_{m}'], P[f'v_{m}_HA'] = ok_l, h_l, ha_l
    return P

def minicheck_cols(P, MC, thr):
    # 응답 점수 = 문장마다 (1 − 근거 지지 확률)의 최대. 임계값은 J-cal 멀쩡한 응답 등각 10%
    P['s_minicheck'] = [max([1 - p for p in MC[str(i)]['p']], default=0.0) for i in P.i]
    P['v_minicheck'] = P.s_minicheck >= thr
    return P

import anthropic
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ')
A = anthropic.Anthropic(api_key=get_key('ANTHROPIC_API_KEY'), max_retries=3)
PRICE = {'claude-haiku-5-5': (0.05, 0.25), 'claude-sonnet-5-5': (1.0, 5.0), 'claude-opus-5-5': (2.0, 10.0)}   # 배치 $/백만 토큰 (2026-11)

def text_of(m): return ''.join(b.text for b in m.content if getattr(b, 'type', '') == 'text')
def rec(m): return {'text': text_of(m), 'stop': m.stop_reason, 'model': m.model, 'in': m.usage.input_tokens, 'out': m.usage.output_tokens}
def cost(res): return sum(r['in'] / 1e6 * PRICE.get(r['model'], (2, 10))[0] + r['out'] / 1e6 * PRICE.get(r['model'], (2, 10))[1] for r in res.values())

def run_batch(reqs, path, chunk=400):
    # reqs: cid -> 요청. 결과: cid -> rec. 다시 실행하면 제출한 묶음·받은 결과는 건너뛴다
    res = load_json(path) or {}
    bpath = path.replace('.json', '_batches.json')
    B = load_json(bpath) or {'chunks': []}
    sent = {c for ch in B['chunks'] for c in ch['cids']}
    todo = [c for c in reqs if c not in sent and c not in res]
    for k in range(0, len(todo), chunk):
        part = todo[k:k + chunk]
        b = A.messages.batches.create(requests=[{'custom_id': c, 'params': reqs[c]} for c in part])
        B['chunks'].append({'id': b.id, 'cids': part, 'done': False}); save_json(B, bpath)
        print(f'  제출 {b.id} ({len(part)}건)', flush=True)
    for ch in B['chunks']:
        if ch.get('done'): continue
        while True:
            b = A.messages.batches.retrieve(ch['id']); rc = b.request_counts
            print(f'  {time.strftime("%H:%M:%S")} {ch["id"][-8:]} {b.processing_status} | 처리 중 {rc.processing}, 성공 {rc.succeeded}, 오류 {rc.errored}', flush=True)
            if b.processing_status == 'ended': break
            time.sleep(60)
        for r in A.messages.batches.results(ch['id']):
            if r.result.type == 'succeeded': res[r.custom_id] = rec(r.result.message)
        ch['done'] = True; save_json(B, bpath); save_json(res, path)
    miss = [c for c in reqs if c not in res]
    for c in miss[:100]:                                   # 배치 실패분은 바로 요청 1회
        try: res[c] = rec(A.messages.create(**reqs[c]))
        except Exception as e: print(f'  {c}: {type(e).__name__} {str(e)[:150]}'); break
    save_json(res, path)
    print(f'  결과 {len(res)}/{len(reqs)} | 사용량 약 ${cost({c: res[c] for c in reqs if c in res}):.2f}')
    return res


REQ = fb_requests(P, FB)
dm_files = [os.path.join(K2_DIR, f'{k}_devM.json') for k in ['cases', 'gen', 'label']]
HAS_DM = all(os.path.exists(f) for f in dm_files)
if HAS_DM:
    CM, GM, LM = (load_json(f) for f in dm_files)
    DM_VALID = [c for c in CM if gen_status(GM.get(c)) == 'valid' and c in LM]
    REQ_DM = {m: {f'{m}-{c}': v_params(CM[c], GM[c]['text'], model) for c in DM_VALID} for m, model in CANDS_CLAUDE.items()}
else:
    REQ_DM = {m: {} for m in CANDS_CLAUDE}
EST = sum(est_cost(r) for r in REQ.values()) + sum(est_cost(r) for r in REQ_DM.values())
print(f'요청: FaithBench {sum(map(len, REQ.values()))}, devM {sum(map(len, REQ_DM.values()))} | 예상 비용 ${EST:.2f} (상한 ${BUDGET_USD})')
assert EST <= BUDGET_USD, '예상 비용이 상한을 넘음 — 멈춤'
_i = int(P.i.iloc[0])                                  # 형식 확인 1건 (실시간, 1센트 미만)
for m, model in CANDS_CLAUDE.items():
    _r = rec(A.messages.create(**v_params(fb_case(FB[_i]), FB[_i]['summary'], model)))
    _labs, _ok = parse(_r['text'], len(sentences(FB[_i]['summary'])))
    print(f"  형식 확인 {m}: 파싱 {'성공' if _ok else '실패'} {[v['label'] for v in _labs.values()]} | 입력 {_r['in']} 출력 {_r['out']}")
RES = {m: run_batch(REQ[m], os.path.join(J1_DIR, f'v_{m}_fb.json')) for m in CANDS_CLAUDE}
P = claude_cols(P, FB, RES)
for m in CANDS_CLAUDE:
    print(f'  {m}: 파싱 성공 J-cal {P[P.part == "cal"][f"ok_{m}"].mean():.3f}, J-test {P[P.part == "test"][f"ok_{m}"].mean():.3f}')

## 4. MiniCheck-7B 후보 (설치·실행이 안 되면 후보에서 빠진다)

In [ ]:
MC = None
try:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'minicheck[llm] @ git+https://github.com/Liyan06/MiniCheck.git@main'], check=True)
    from minicheck.minicheck import MiniCheck
    mc = MiniCheck(model_name='Bespoke-MiniCheck-7B', enable_prefix_caching=False, cache_dir='/content/ckpts')
    mp = os.path.join(J1_DIR, 'minicheck_fb.json'); MC = load_json(mp) or {}
    t0 = time.time()
    for k, i in enumerate(P.i):
        if str(i) in MC: continue
        ss = [t for _, _, t in sentences(FB[i]['summary'])]
        t1 = time.perf_counter()
        _, prob, _, _ = mc.score(docs=[FB[i]['source']] * len(ss), claims=ss)
        MC[str(i)] = {'p': [float(x) for x in prob], 'ms': (time.perf_counter() - t1) * 1000}
        if (k + 1) % 50 == 0: save_json(MC, mp); print(f'  {k + 1}/{len(P)} ({time.time() - t0:.0f}s)', flush=True)
    save_json(MC, mp)
except Exception as e:
    print('MiniCheck 실행 안 됨 → 후보에서 뺀다:', type(e).__name__, str(e)[:300]); MC = None
if MC is not None:
    _cal = P[P.part == 'cal'].copy()
    _cal['s'] = [max([1 - p for p in MC[str(i)]['p']], default=0.0) for i in _cal.i]
    THR_MC = conformal(_cal[~_cal.hal].s.values, 0.10)
    P = minicheck_cols(P, MC, THR_MC); print(f'MiniCheck 임계값(J-cal 멀쩡한 응답 등각 10%) {THR_MC:.4f}')

## 5. 선택(J-cal) → 판정(J-test)

In [ ]:
def rates(d, col):
    return float(d[d.hal][col].astype(bool).mean()), float(d[~d.hal][col].astype(bool).mean())

def select(P, cands):
    # J-cal에서: 멀쩡한 응답 표시율 ≤ 15%인 후보 중 탐지율 최대 (동점이면 ORDER 순)
    C = P[P.part == 'cal']; ok = []
    for c in cands:
        det, fl = rates(C, f'v_{c}')
        print(f'  J-cal {c:10} 탐지 {det:.3f} | 멀쩡한 응답 표시 {fl:.3f}')
        if fl <= 0.15: ok.append((det, -ORDER.index(c), c))
    return max(ok)[2] if ok else None

def gate_threshold(P, keep=0.95):
    # 1단(배포 조합) 통과 기준: J-cal 환각 응답의 95%가 2단으로 넘어가는 점수
    h = np.sort(P[(P.part == 'cal') & P.hal].max_c.values)
    return float(h[int(np.floor((1 - keep) * len(h)))])

def judge_j1(P, sel, seed=2035):
    T = P[P.part == 'test']
    out = {'n_test': len(T), 'n_test_hal': int(T.hal.sum()), 'n_test_clean': int((~T.hal).sum()), 'selected': sel}
    out['cB'] = dict(zip(['det', 'flag'], rates(T, 'flag_cB')))
    if sel is None:
        out.update(J1_1=False, J1_2=False, J1=False); return out
    v = f'v_{sel}'
    det, fl = rates(T, v)
    out['det'], out['flag'] = det, fl
    out['det_ci'] = cluster_boot(T, lambda d: d[d.hal][v].astype(bool).mean(), seed=seed)
    out['flag_ci'] = cluster_boot(T, lambda d: d[~d.hal][v].astype(bool).mean(), seed=seed)
    diff = lambda d: d[d.hal][v].astype(bool).mean() - d[d.hal].flag_cB.astype(bool).mean()
    out['diff_vs_cB'] = {'delta': det - out['cB']['det'], 'ci': cluster_boot(T, diff, seed=seed)}
    out['J1_1'] = bool(det >= 0.50 and out['diff_vs_cB']['ci'][0] > 0)
    out['J1_2'] = bool(fl <= 0.15)
    out['J1'] = out['J1_1'] and out['J1_2']
    k = f'k_{sel}'
    out['cascade'] = {'det': rates(T, k)[0], 'flag': rates(T, k)[1], 'escalate': float(T.gate.mean())}
    out['J1_3'] = bool(out['cascade']['det'] >= det - 0.03)
    return out

CANDS = [m for m in CANDS_CLAUDE
         if P[P.part == 'cal'][f'ok_{m}'].mean() >= 0.95 and P[P.part == 'test'][f'ok_{m}'].mean() >= 0.95]
if MC is not None: CANDS.append('minicheck')
CANDS = sorted(CANDS, key=ORDER.index)
print('J1-0 통과 후보:', CANDS)
TAU1 = gate_threshold(P)
P['gate'] = P.max_c >= TAU1
for c in CANDS: P[f'k_{c}'] = P.gate & P[f'v_{c}']
SEL = select(P, CANDS)
R = judge_j1(P, SEL)
T = P[P.part == 'test']
print(f"\nJ-test: 응답 {R['n_test']} (환각 {R['n_test_hal']}, 멀쩡 {R['n_test_clean']}) | 배포 조합 B: 탐지 {R['cB']['det']:.3f}, 표시 {R['cB']['flag']:.3f}")
for c in CANDS:
    d, f = rates(T, f'v_{c}'); print(f'  (보고) J-test {c:10} 탐지 {d:.3f} | 표시 {f:.3f}' + (f" | H 또는 A: {rates(T, f'v_{c}_HA')[0]:.3f} / {rates(T, f'v_{c}_HA')[1]:.3f}" if c in CANDS_CLAUDE else ''))
if SEL is None:
    print('\n=== J1 미달: J-cal에서 멀쩡한 응답 표시율 ≤ 15%인 후보가 없음 ===')
else:
    dc = R['diff_vs_cB']
    print(f"\n=== 선택: {SEL} ===")
    print(f"=== J1-1(주): 탐지율 {R['det']:.3f} [{R['det_ci'][0]:.3f}, {R['det_ci'][1]:.3f}] (≥ 0.50), 배포 조합 B 대비 {dc['delta']:+.3f} [{dc['ci'][0]:+.3f}, {dc['ci'][1]:+.3f}] (하한 > 0) → {'충족' if R['J1_1'] else '미달'} ===")
    print(f"=== J1-2: 멀쩡한 응답 표시율 {R['flag']:.3f} [{R['flag_ci'][0]:.3f}, {R['flag_ci'][1]:.3f}] (≤ 0.15) → {'충족' if R['J1_2'] else '미달'} ===")
    print(f"=== J1 {'채택' if R['J1'] else '미채택'} ===")
    cz = R['cascade']
    print(f"=== J1-3: 1단 거르기(조합 ≥ {TAU1:.3f}, 2단으로 보내는 비율 {cz['escalate']:.3f}) + 2단 탐지 {cz['det']:.3f} vs 2단 단독 {R['det']:.3f} → {'거르기 사용' if R['J1_3'] else '모든 응답 2단'} ===")
print('(보고) 생성 모델별 J-test 탐지율:', {m.split('/')[-1]: round(float(d[d.hal][f'v_{SEL}'].mean()), 3) for m, d in T.groupby('summarizer')} if SEL else '-')

## 6. J1-4 지연 — 선택 후보의 응답 1개 검증 시간

In [ ]:
LAT = None
if SEL in CANDS_CLAUDE:
    ids = list(P[P.part == 'cal'].i)[:40]; ms = []
    for i in ids:                                     # 실시간 요청 (배치 아님, 표준 단가)
        t1 = time.perf_counter(); A.messages.create(**v_params(fb_case(FB[i]), FB[i]['summary'], CANDS_CLAUDE[SEL]))
        ms.append((time.perf_counter() - t1) * 1000)
    LAT = ms
elif SEL == 'minicheck':
    LAT = [MC[str(i)]['ms'] for i in P[P.part == 'cal'].i]
if LAT:
    p50, p95 = np.percentile(LAT, 50), np.percentile(LAT, 95)
    R['latency_ms'] = {'n': len(LAT), 'p50': float(p50), 'p95': float(p95)}; R['J1_4'] = bool(p95 <= 5000)
    print(f"=== J1-4: {SEL} 응답 1개 검증 p50 {p50:.0f}ms, p95 {p95:.0f}ms (≤ 5000) → {'hold 가능' if R['J1_4'] else '응답 뒤 경고(비동기)만'} ===")

## 7. (보고) devM — 최신 Claude 응답, Opus 정답 (Claude 후보는 순환이므로 판정에 쓰지 않음)

In [ ]:
DM = None
if HAS_DM:
    gold = {}
    for c in DM_VALID:
        labs, ok = parse(LM[c]['text'], len(sentences(GM[c]['text'])))
        if ok: gold[c] = 'H' in [v['label'] for v in labs.values()]
    DM = {'n': len(gold), 'n_hal': sum(gold.values())}
    for m, model in CANDS_CLAUDE.items():
        res = run_batch(REQ_DM[m], os.path.join(J1_DIR, f'v_{m}_devM.json'))
        fl = {}
        for c in gold:
            r = res.get(f'{m}-{c}'); labs, ok = parse(r['text'], len(sentences(GM[c]['text']))) if r else ({}, False)
            fl[c] = ok and 'H' in [v['label'] for v in labs.values()]
        h = [fl[c] for c in gold if gold[c]]; k = [fl[c] for c in gold if not gold[c]]
        DM[m] = {'det': float(np.mean(h)), 'flag': float(np.mean(k))}
        print(f'  devM {m:7} 탐지 {DM[m]["det"]:.3f} | 멀쩡한 응답 표시 {DM[m]["flag"]:.3f}  (환각 {len(h)}, 멀쩡 {len(k)})')
else:
    print('k2/의 cases·gen·label_devM.json이 없어 건너뜀')

## 8. 저장

In [ ]:
usage = {f: round(cost(load_json(os.path.join(J1_DIR, f))), 2) for f in os.listdir(J1_DIR) if f.startswith('v_') and f.endswith('.json') and '_batches' not in f}
print('API 사용량(배치 단가 기준, 지연 측정 제외) $', usage, '합계', round(sum(usage.values()), 2))
LOCAL = '/content/j1_local'; os.makedirs(LOCAL, exist_ok=True)
cols = [c for c in P.columns if c in ('i', 'src', 'summarizer', 'hal', 'part', 'max_c', 'flag_cB', 'gate', 's_minicheck') or c.startswith(('ok_', 'v_', 'k_'))]
P[cols].to_csv(os.path.join(LOCAL, 'j1_rows.csv'), index=False)        # 원문 없음
json.dump(dict(R, cands=CANDS, tau1=TAU1, thr_minicheck=(THR_MC if MC is not None else None), devM=DM, cost_usd=usage, gpu=GPU),
          open(os.path.join(LOCAL, 'j1_summary.json'), 'w'), ensure_ascii=False, indent=1, default=float)
for f in os.listdir(LOCAL):
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(J1_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
zp = shutil.make_archive('/content/j1_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)